# Week 9 Training (Colab)

This notebook trains Week 9 artifacts from `auction_snapshots.csv` (teammate output) **without changing snapshot generation logic**.

Outputs:
- `models/point_model.pkl`
- `models/week9_metrics.json`
- `models/metadata.json`


In [2]:
# Install dependencies (Colab)
!pip -q install numpy pandas scikit-learn

In [3]:
import json
import pickle
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from google.colab import files


Upload Kush's auction_snapshots.csv

In [4]:
# Upload teammate snapshot CSV
# Expected file: auction_snapshots.csv
uploaded = files.upload()
if len(uploaded) == 0:
    raise RuntimeError("No file uploaded.")

INPUT_CSV = next(iter(uploaded.keys()))
print(f"Using input file: {INPUT_CSV}")


Saving auction_snapshots.csv to auction_snapshots.csv
Using input file: auction_snapshots.csv


In [5]:
TARGET_COLUMN = "final_price"
GROUP_COLUMN = "auction_id"

REQUIRED_COLUMNS = [
    "auction_id",
    "item_name",
    "auction_type",
    "auction_days",
    "snapshot_pct",
    "snapshot_time_days",
    "estimated_bid_amount",
    "opening_bid",
    "final_price",
    "bid_count_so_far",
    "unique_bidders_so_far",
    "max_observed_bid_so_far",
    "leading_bidder_so_far",
    "leading_bidder_rate_so_far",
]

CATEGORICAL_COLUMNS = ["item_name", "auction_type", "snapshot_pct"]
NUMERIC_COLUMNS = [
    "auction_days",
    "snapshot_time_days",
    "opening_bid",
    "bid_count_so_far",
    "unique_bidders_so_far",
    "max_observed_bid_so_far",
    "leading_bidder_rate_so_far",
]
FEATURE_COLUMNS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS


In [6]:
def metric_dict(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {"mae": mae, "rmse": rmse}


def grouped_metrics(y_true, y_pred, groups):
    out = {}
    grp = pd.Series(groups).astype(str)
    for value in sorted(grp.unique(), key=str):
        idx = grp == value
        if int(idx.sum()) == 0:
            continue
        out[str(value)] = metric_dict(y_true[idx], y_pred[idx])
    return out


def load_and_validate(csv_path):
    df = pd.read_csv(csv_path)

    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    numeric_to_parse = [
        "auction_days",
        "snapshot_pct",
        "snapshot_time_days",
        "estimated_bid_amount",
        "opening_bid",
        "final_price",
        "bid_count_so_far",
        "unique_bidders_so_far",
        "max_observed_bid_so_far",
        "leading_bidder_rate_so_far",
    ]
    for col in numeric_to_parse:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Keep stage label for reporting/modeling as categorical string.
    df["snapshot_pct"] = df["snapshot_pct"].map(lambda x: f"{x:.2f}" if pd.notna(x) else "MISSING")

    for col in ["item_name", "auction_type", "leading_bidder_so_far"]:
        df[col] = df[col].astype("string").fillna("MISSING").astype(str)

    df[GROUP_COLUMN] = df[GROUP_COLUMN].astype("string").fillna("MISSING").astype(str)

    before = len(df)
    df = df.dropna(subset=[TARGET_COLUMN]).copy()
    dropped = before - len(df)
    if dropped:
        print(f"Dropped {dropped} rows with missing target.")

    return df


def make_split(df, test_size=0.2, random_state=42):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(df, df[TARGET_COLUMN], groups=df[GROUP_COLUMN]))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    overlap = set(train_df[GROUP_COLUMN]).intersection(set(test_df[GROUP_COLUMN]))
    if overlap:
        raise RuntimeError(f"Leakage guard failed: {len(overlap)} overlapping auction IDs")

    return train_df, test_df


def baseline_item_median(train_df, test_df):
    global_median = float(train_df[TARGET_COLUMN].median())
    by_item = train_df.groupby("item_name")[TARGET_COLUMN].median().to_dict()
    return test_df["item_name"].map(by_item).fillna(global_median).to_numpy(dtype=float)


def baseline_opening_linear(train_df, test_df):
    med_open = float(train_df["opening_bid"].median())
    x_train = train_df[["opening_bid"]].fillna(med_open).to_numpy(dtype=float)
    x_test = test_df[["opening_bid"]].fillna(med_open).to_numpy(dtype=float)
    y_train = train_df[TARGET_COLUMN].to_numpy(dtype=float)

    model = LinearRegression()
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    return np.clip(pred, a_min=0.0, a_max=None)


def baseline_current_price(train_df, test_df):
    global_median = float(train_df[TARGET_COLUMN].median())
    pred = test_df["max_observed_bid_so_far"].where(
        test_df["max_observed_bid_so_far"].notna(), test_df["opening_bid"]
    )
    pred = pred.fillna(global_median).to_numpy(dtype=float)
    return np.clip(pred, a_min=0.0, a_max=None)


def train_point_model(train_df, random_state=42, n_estimators=300):
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                CATEGORICAL_COLUMNS,
            ),
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                ]),
                NUMERIC_COLUMNS,
            ),
        ],
        remainder="drop",
    )

    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        random_state=random_state,
        n_jobs=-1,
        min_samples_leaf=1,
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", rf),
    ])

    x_train = train_df[FEATURE_COLUMNS]
    y_train = train_df[TARGET_COLUMN].to_numpy(dtype=float)
    pipeline.fit(x_train, y_train)
    return pipeline


def aggregate_raw_feature_importance(model_pipeline):
    pre = model_pipeline.named_steps["preprocessor"]
    model = model_pipeline.named_steps["model"]

    names = pre.get_feature_names_out()
    importances = model.feature_importances_
    agg = {name: 0.0 for name in FEATURE_COLUMNS}

    for encoded_name, imp in zip(names, importances):
        if encoded_name.startswith("num__"):
            raw = encoded_name.replace("num__", "", 1)
        elif encoded_name.startswith("cat__"):
            tail = encoded_name.replace("cat__", "", 1)
            raw = None
            for col in CATEGORICAL_COLUMNS:
                prefix = f"{col}_"
                if tail == col or tail.startswith(prefix):
                    raw = col
                    break
            if raw is None:
                raw = tail.split("_", 1)[0]
        else:
            raw = encoded_name

        agg[raw] = float(agg.get(raw, 0.0) + float(imp))

    return dict(sorted(agg.items(), key=lambda kv: kv[1], reverse=True))


In [7]:
# Train Week 9 models and baselines
df = load_and_validate(INPUT_CSV)
train_df, test_df = make_split(df, test_size=0.2, random_state=42)

print("Rows total:", len(df))
print("Rows train:", len(train_df))
print("Rows test:", len(test_df))
print("Auctions train:", train_df[GROUP_COLUMN].nunique())
print("Auctions test:", test_df[GROUP_COLUMN].nunique())

# Leakage guard
overlap = set(train_df[GROUP_COLUMN]).intersection(set(test_df[GROUP_COLUMN]))
print("Train/Test auction overlap:", len(overlap))

y_test = test_df[TARGET_COLUMN].to_numpy(dtype=float)

pred_item = baseline_item_median(train_df, test_df)
pred_open = baseline_opening_linear(train_df, test_df)
pred_curr = baseline_current_price(train_df, test_df)

point_model = train_point_model(train_df, random_state=42, n_estimators=300)
pred_rf = point_model.predict(test_df[FEATURE_COLUMNS])

all_preds = {
    "item_median_baseline": pred_item,
    "opening_bid_linear_baseline": pred_open,
    "current_price_heuristic_baseline": pred_curr,
    "random_forest_point_model": pred_rf,
}

overall_rows = []
for name, pred in all_preds.items():
    m = metric_dict(y_test, pred)
    overall_rows.append({"model": name, **m})

overall_df = pd.DataFrame(overall_rows).sort_values("mae")
overall_df


Rows total: 4396
Rows train: 3514
Rows test: 882
Auctions train: 502
Auctions test: 126
Train/Test auction overlap: 0


,model,mae,rmse
3,random_forest_point_model,71.919174,193.077933
2,current_price_heuristic_baseline,80.733209,171.756374
0,item_median_baseline,158.069802,454.892711
1,opening_bid_linear_baseline,177.401613,367.485198


In [8]:
# Metrics by stage and by item
by_stage = {}
by_item = {}
for name, pred in all_preds.items():
    by_stage[name] = grouped_metrics(y_test, pred, test_df["snapshot_pct"].to_numpy())
    by_item[name] = grouped_metrics(y_test, pred, test_df["item_name"].to_numpy())

print("By snapshot_pct (Random Forest):")
pd.DataFrame(by_stage["random_forest_point_model"]).T


By snapshot_pct (Random Forest):


,mae,rmse
0.25,84.134446,200.281327
0.50,82.936598,211.762878
0.75,72.002863,198.494939
0.85,71.940363,189.158684
0.90,71.756433,189.733073
0.95,66.679834,182.320436
1.00,53.983680,177.698799


In [9]:
print("By item_name (Random Forest):")
pd.DataFrame(by_item["random_forest_point_model"]).T


By item_name (Random Forest):


,mae,rmse
Cartier wristwatch,258.086337,399.461555
Palm Pilot M515 PDA,10.251439,14.552647
Xbox game console,29.682191,42.995903


In [ ]:
# Save Week 9 artifacts
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

feature_importance_raw = aggregate_raw_feature_importance(point_model)
numeric_medians = train_df[NUMERIC_COLUMNS].median().replace({np.nan: None}).to_dict()
numeric_medians = {k: (float(v) if v is not None and pd.notna(v) else None) for k, v in numeric_medians.items()}

metrics = {
    "dataset": {
        "input_csv": str(Path(INPUT_CSV).resolve()),
        "rows_total": int(len(df)),
        "rows_train": int(len(train_df)),
        "rows_test": int(len(test_df)),
        "auctions_total": int(df[GROUP_COLUMN].nunique()),
        "auctions_train": int(train_df[GROUP_COLUMN].nunique()),
        "auctions_test": int(test_df[GROUP_COLUMN].nunique()),
        "snapshot_pct_values": sorted([str(v) for v in df["snapshot_pct"].dropna().unique()]),
    },
    "leakage_check": {
        "train_test_auction_overlap_count": int(len(set(train_df[GROUP_COLUMN]).intersection(set(test_df[GROUP_COLUMN]))))
    },
    "overall": {
        name: metric_dict(y_test, pred) for name, pred in all_preds.items()
    },
    "by_snapshot_pct": by_stage,
    "by_item_name": by_item,
}

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(Path(INPUT_CSV).resolve()),
    "feature_columns": FEATURE_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "numeric_columns": NUMERIC_COLUMNS,
    "target_column": TARGET_COLUMN,
    "group_column": GROUP_COLUMN,
    "snapshot_stage_column": "snapshot_pct",
    "numeric_feature_medians": numeric_medians,
    "feature_importance_raw": feature_importance_raw,
    "limitations": [
        "Built from historical sold-auction snapshots only.",
        "No seller reputation, item condition text, images, or calendar-time context.",
        "Proxy-bidding behavior may make bidding dynamics appear non-monotonic."
    ]
}

model_artifact = {
    "pipeline": point_model,
    "feature_columns": FEATURE_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "numeric_columns": NUMERIC_COLUMNS,
    "target_column": TARGET_COLUMN,
}

with open(models_dir / "point_model.pkl", "wb") as f:
    pickle.dump(model_artifact, f)

with open(models_dir / "week9_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

with open(models_dir / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
print(" -", (models_dir / "point_model.pkl").resolve())
print(" -", (models_dir / "week9_metrics.json").resolve())
print(" -", (models_dir / "metadata.json").resolve())


In [ ]:
# Minimal Week 9 point prediction interface (in-notebook)
def predict_point(snapshot_record, model_path="models/point_model.pkl", metadata_path="models/metadata.json"):
    with open(model_path, "rb") as f:
        model_artifact = pickle.load(f)
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    feature_columns = model_artifact["feature_columns"]
    pipeline = model_artifact["pipeline"]

    row = {feature: snapshot_record.get(feature) for feature in feature_columns}
    frame = pd.DataFrame([row])
    point_estimate = float(pipeline.predict(frame)[0])

    # Lightweight factor summaries (Week 9 MVP)
    importance = metadata.get("feature_importance_raw", {})
    ranked = [k for k, _ in sorted(importance.items(), key=lambda kv: kv[1], reverse=True)]
    top_positive = [f"{feat} is an important signal in this estimate." for feat in ranked[:3]]
    top_negative = ["Detailed downside decomposition starts in Week 11 interpretability work."] * 3

    return {
        "point_estimate": round(point_estimate, 4),
        "top_positive_factors": top_positive,
        "top_negative_factors": top_negative,
        "limitations": metadata.get("limitations", []),
    }

sample = test_df.iloc[0]
sample_input = {c: sample[c] for c in FEATURE_COLUMNS}
out = predict_point(sample_input)
out


In [ ]:
# Optional: download artifacts from Colab
files.download("models/point_model.pkl")
files.download("models/week9_metrics.json")
files.download("models/metadata.json")
